In [10]:
import mlrun

from dotenv import load_dotenv
# Loads AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and MLRUN_AWS_ROLE_ARN
load_dotenv() 

import os
# https://docs.mlrun.org/en/stable/store/datastore.html#s3
# print(os.environ['AWS_ACCESS_KEY_ID'])
# print(os.environ['AWS_SECRET_ACCESS_KEY'])
# print(os.environ['MLRUN_AWS_ROLE_ARN'])

from pathlib import Path
from datetime import datetime

artifact_path = Path.cwd().parent
artifact_path = str(artifact_path.as_posix()) # convert windows path to unix path
artifact_path = "file://" + artifact_path
print(artifact_path)
p = mlrun.set_environment("http://localhost:8080", artifact_path=artifact_path)

file://c:/Work/Folder_1/Project_folder/LLM_project_3_FineTune_MLOps/Finetune-legal-llm-mlops


In [11]:
project = mlrun.load_project(name='finetune-legal-extractor', context="../") # project yaml must be in this directory

# Verify it loaded correctly by checking its status or printing the config
# print(project.to_yaml())

In [12]:
system_prompt = """
You are a legal contract analyst assistant that analyzes a legal contract to confirm or deny the status of a set of hypotheses about the contract. Each hypothesis is a statement that may be true or false or neutral based on the content of the contract. Read the whole contract and look for phrases or quotes within it to label each hypothesis with entailment for true, contradidiction for false, and not_mentioned if the hypothesis is neutral and cannot be confirmed or denied. Perform the analysis for all of the 17 hypotheses from nda-1 to nda-20. The hypotheses with their corresponding ids are as follows:

nda-1: All Confidential Information shall be expressly identified by the Disclosing Party.
nda-2: Confidential Information shall only include technical information.
nda-3: Confidential Information may include verbally conveyed information.
nda-4: Receiving Party shall not use any Confidential Information for any purpose other than the purposes stated in Agreement.
nda-5: Receiving Party may share some Confidential Information with some of Receiving Party's employees.
nda-7: Receiving Party may share some Confidential Information with some third-parties (including consultants, agents and professional advisors).
nda-8: Receiving Party shall notify Disclosing Party in case Receiving Party is required by law, regulation or judicial process to disclose any Confidential Information.
nda-10: Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.
nda-11: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.
nda-12: Receiving Party may independently develop information similar to Confidential Information.
nda-13: Receiving Party may acquire information similar to Confidential Information from a third party.
nda-15: Agreement shall not grant Receiving Party any right to Confidential Information.
nda-16: Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.
nda-17: Receiving Party may create a copy of some Confidential Information in some circumstances.
nda-18: Receiving Party shall not solicit some of Disclosing Party's representatives.
nda-19: Some obligations of Agreement may survive termination of Agreement.
nda-20: Receiving Party may retain some Confidential Information even after the return or destruction of Confidential Information.

Read the contract and respond with a structured JSON document that contains a list of 17 JSON objects for each hypothesis. Each object containins the id of the hypothesis, the hypothesis statement, the quotes or phrases from the contract that justifies the label given (this must only come from the legal contract from the user), and the label (entailment, contradiction, not_mentioned). For hypotheses that are labeled as not_mentioned leave the source clause field blank. Only respond with the JSON document, do not provide any additional information, and do not make up your own quotes to fill in source_clause.

Remember:
If label is entailment or contradiction. Fill in source_clause with the quote the justifies the label
If label is neutral. Leave source_clause blank because it cannot be confirmed or denied based on the content of the contract

Follow this format for each label
[
{
    "hypothesis_id:...
    "hypothesis":...
    "source_clause": "Placeholder text"
    "label": "entailment"
},
{
    "hypothesis_id:...
    "hypothesis":...
    "source_clause": "Placeholder text"
    "label": "contradiction"
},
{
    "hypothesis_id:...
    "hypothesis":...
    "source_clause": ""
    "label": "not_mentioned"
},
]
"""

In [24]:
prompt_template=[
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": "{contract}",
    }
]

tag = datetime.now().strftime("%Y%m%d_%H%M") # this is the version
key = "contract_extractor_prompt"
print(tag)

project.log_llm_prompt(
    key=key,
    prompt_template=prompt_template,
    prompt_legend={
        "issue_description": {
            "field": "contract",
            "description": "The legal contract to extract data from",
        },
    },
    invocation_config={
        "temperature": 0.2,
        "top_p": 0.95
    },
    description="Prompt template for the legal extractor",
    artifact_path=f"s3://legal-llama-data/llm_prompt/{key}/{tag}",#artifact_path + f"/prompt/{key}/{tag}",
    tag=tag
)


20260412_2201


In [14]:
project.save()

In [15]:
# Retrieving the prompt
prompt = project.list_llm_prompts(name="contract_extractor_prompt", tag="latest")[0].read_prompt()
print(type(prompt[0]))
print(prompt)

<class 'dict'>
[{'role': 'system', 'content': '\nYou are a legal contract analyst assistant that analyzes a legal contract to confirm or deny the status of a set of hypotheses about the contract. Each hypothesis is a statement that may be true or false or neutral based on the content of the contract. Read the whole contract and look for phrases or quotes within it to label each hypothesis with entailment for true, contradidiction for false, and not_mentioned if the hypothesis is neutral and cannot be confirmed or denied. Perform the analysis for all of the 17 hypotheses from nda-1 to nda-20. The hypotheses with their corresponding ids are as follows:\n\nnda-1: All Confidential Information shall be expressly identified by the Disclosing Party.\nnda-2: Confidential Information shall only include technical information.\nnda-3: Confidential Information may include verbally conveyed information.\nnda-4: Receiving Party shall not use any Confidential Information for any purpose other than th

In [16]:
project.get_artifact(key="contract_extractor_prompt", tag="latest").read_prompt()

[{'role': 'system',
  'content': '\nYou are a legal contract analyst assistant that analyzes a legal contract to confirm or deny the status of a set of hypotheses about the contract. Each hypothesis is a statement that may be true or false or neutral based on the content of the contract. Read the whole contract and look for phrases or quotes within it to label each hypothesis with entailment for true, contradidiction for false, and not_mentioned if the hypothesis is neutral and cannot be confirmed or denied. Perform the analysis for all of the 17 hypotheses from nda-1 to nda-20. The hypotheses with their corresponding ids are as follows:\n\nnda-1: All Confidential Information shall be expressly identified by the Disclosing Party.\nnda-2: Confidential Information shall only include technical information.\nnda-3: Confidential Information may include verbally conveyed information.\nnda-4: Receiving Party shall not use any Confidential Information for any purpose other than the purposes st

In [17]:
x = project.get_artifact(key="contract_extractor_prompt", tag="latest").read_prompt()
x[1]['content']

'{contract}'

In [18]:
# # mlrun.set_environment("http://localhost:8080")
# db = mlrun.get_run_db()

# db.del_artifact(
#     key="contract_extractor_prompt",
#     tag="20260412_1810",
#     project='finetune-legal-extractor',
#     deletion_strategy="data-force"  # Cascades deletion to underlying data
# )